In [1]:
### IMPORT ###

import pandas as pd
from pandas import IndexSlice as idx

import scipy.io as io
import scipy.sparse as sprs

import numpy as np

import os

In [4]:
### INPUTS ###

input_folder = './input/'                 # folder with parameters
output_folder =  './results/'             # folder to write results to
losses = './evaluation/'                  # folder to store the refined results to
os.makedirs(losses, exist_ok=True)

a_shock = 'Afghanistan'

In [5]:
### LOADING DATA ###

# Load information
io_codes = pd.read_csv(input_folder+'io_codes_alph.csv').drop('Unnamed: 0', axis = 1)
su_codes = pd.read_csv(input_folder+'su_codes_alph.csv').drop('Unnamed: 0', axis = 1)

# Create single indexes
areas = np.array(sorted(set(io_codes['area'])))
items = np.array(sorted(set(io_codes['item'])))
processes = np.array(sorted(set(su_codes['proc'])))

# Create multi indexes
ai_index = pd.MultiIndex.from_product([areas, items])
ap_index = pd.MultiIndex.from_product([areas, items])

# Load  further information on countries
a_frame = pd.read_csv(input_folder+'a_frame.csv', index_col='area')

# Load the result of the shocked simulation
X = pd.read_csv(output_folder+'base.csv', index_col=[0,1], header=[0])
XS_comp = pd.read_csv(output_folder+a_frame.loc[a_shock,'code']+'_comp.csv', index_col=[0,1], header=[0,1])
XS_no_comp = pd.read_csv(output_folder+a_frame.loc[a_shock,'code']+'_no_comp.csv', index_col=[0,1], header=[0,1])

In [6]:
### COMPUTATIONS ###

# Compute relative loss
RL_no_comp = XS_no_comp.copy()
RL_comp = XS_comp.copy()
for col in XS_no_comp.columns:
    RL_no_comp[col] = ((X['base'] - RL_no_comp[col])/X['base']).fillna(0)
    RL_comp[col] = ((X['base'] - RL_comp[col])/X['base']).fillna(0)  
RL_no_comp[RL_no_comp < -1] = -1
RL_comp[RL_comp < -1] = -1

# Setup a dataframe for the relative loss
RL_no_comp.columns = pd.MultiIndex.from_product([[a_shock],items])
RL_no_comp.columns.names = ['a_shock','i_shock']
RL_no_comp.index.names = ['a_receive','i_receive'] 
RL_comp.columns = pd.MultiIndex.from_product([[a_shock],items])
RL_comp.columns.names = ['a_shock','i_shock']
RL_comp.index.names = ['a_receive','i_receive'] 

# Save
RL_no_comp.to_csv(losses+'RL-'+a_frame.loc[a_shock,'code']+'_no_comp.csv')
RL_comp.to_csv(losses+'RL-'+a_frame.loc[a_shock,'code']+'_comp.csv')


# Compute absolute loss
AL_no_comp = XS_no_comp.copy()
AL_comp = XS_comp.copy()
for col in XS_no_comp.columns:
    AL_no_comp[col] = X['base'] - XS_no_comp[col]
    AL_comp[col] = X['base'] - XS_comp[col]

# Setup a dataframe for the absolute loss
AL_no_comp.columns = pd.MultiIndex.from_product([[a_shock],items])
AL_no_comp.columns.names = ['a_shock','i_shock']
AL_no_comp.index.names = ['a_receive','i_receive'] 
AL_comp.columns = pd.MultiIndex.from_product([[a_shock],items])
AL_comp.columns.names = ['a_shock','i_shock']
AL_comp.index.names = ['a_receive','i_receive']  

# Save
AL_no_comp.to_csv(losses+'AL-'+a_frame.loc[a_shock,'code']+'_no_comp.csv')
AL_comp.to_csv(losses+'AL-'+a_frame.loc[a_shock,'code']+'_comp.csv')

# Compute absolute loss per capita
pop = a_frame['population']
AL_no_comp_pc = AL_no_comp.div(AL_no_comp.index.get_level_values('a_receive').map(pop), axis=0) * 1000 #times 1000 to transform from tons to kg
AL_comp_pc = AL_comp.div(AL_comp.index.get_level_values('a_receive').map(pop), axis=0) * 1000

# Setup a dataframe for the absolute loss
AL_no_comp_pc.columns = pd.MultiIndex.from_product([[a_shock],items])
AL_no_comp_pc.columns.names = ['a_shock','i_shock']
AL_no_comp_pc.index.names = ['a_receive','i_receive'] 
AL_comp_pc.columns = pd.MultiIndex.from_product([[a_shock],items])
AL_comp_pc.columns.names = ['a_shock','i_shock']
AL_comp_pc.index.names = ['a_receive','i_receive']  

# Save
AL_no_comp_pc.to_csv(losses+'AL_pc-'+a_frame.loc[a_shock,'code']+'_no_comp.csv')
AL_comp_pc.to_csv(losses+'AL_pc-'+a_frame.loc[a_shock,'code']+'_comp.csv')